# 異常検知 — 正常データだけで学習する

**出典**　機械学習 4 分野の教科書 — 第 10 章 3-B 次元削減と異常検知 ／ 07 実装：圧縮して、見て、異常を見つける

**このノートのパス**　`ml-code/10_dimreduction_anomaly/03_anomaly_detection.ipynb`

**このノートで確かめること**
「0 の画像だけが正常」という設定を作り、IsolationForest と OneClassSVM を比べる。
**`score_samples` は「正常らしさ」なので符号を反転して異常スコアにする**——ここを間違えると AUC が 0.5 未満になる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn numpy` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

In [3]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score

RS = 42
X, y = load_digits(return_X_y=True)

# 「0 の画像」だけを正常とみなし、それ以外を異常として扱う設定を作る
normal = X[y == 0]
rng = np.random.default_rng(RS)
anomaly = X[y != 0][rng.choice((y != 0).sum(), size=40, replace=False)]

X_train = normal[:150]                                  # 訓練は正常のみ
X_test = np.vstack([normal[150:], anomaly])
y_test = np.r_[np.zeros(len(normal) - 150), np.ones(len(anomaly))]   # 1 が異常
print("訓練（正常のみ）:", X_train.shape, "／ テスト:", X_test.shape,
      "うち異常", int(y_test.sum()), "件")

for name, model in [
    ("IsolationForest", IsolationForest(contamination=0.1, random_state=RS)),
    ("OneClassSVM",     OneClassSVM(kernel="rbf", gamma="scale", nu=0.1)),
]:
    model.fit(X_train)
    # score_samples は「正常らしさ」なので、符号を反転して「異常スコア」にする
    score = -model.score_samples(X_test)
    print(f"{name:16s} ROC-AUC = {roc_auc_score(y_test, score):.3f}")

訓練（正常のみ）: (150, 64) ／ テスト: (68, 64) うち異常 40 件
IsolationForest  ROC-AUC = 0.997
OneClassSVM      ROC-AUC = 1.000


### 符号を反転し忘れるとどうなるか

In [4]:
iso = IsolationForest(contamination=0.1, random_state=RS).fit(X_train)
print("正しい（反転あり）:", round(roc_auc_score(y_test, -iso.score_samples(X_test)), 3))
print("誤り（反転なし）  :", round(roc_auc_score(y_test,  iso.score_samples(X_test)), 3))
print("\n※ AUC が 0.5 を大きく下回ったら、まず符号を疑うこと")

正しい（反転あり）: 0.997
誤り（反転なし）  : 0.003

※ AUC が 0.5 を大きく下回ったら、まず符号を疑うこと


### 閾値をどう決めるか — ここでも「0.5」に相当する既定値は根拠を持たない

In [5]:
score = -iso.score_samples(X_test)

print(f"{'予算(上位%)':>12} {'閾値':>9} {'検出':>5} {'見逃し':>6} {'誤検知':>6}")
for q in [0.01, 0.05, 0.10, 0.20]:
    th = np.quantile(score, 1 - q)
    pred = (score >= th).astype(int)
    tp = int(((pred == 1) & (y_test == 1)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    print(f"{q:>11.0%} {th:>9.4f} {tp:>5} {fn:>6} {fp:>6}")

     予算(上位%)        閾値    検出    見逃し    誤検知
         1%    0.7488     1     39      0
         5%    0.7304     4     36      0
        10%    0.6887     7     33      0
        20%    0.6737    14     26      0


## 読み取りどころ

- **訓練に使うのは正常データだけ。**異常のラベルは**評価にしか使っていない**。これが異常検知と不均衡分類の決定的な違いで、**異常の例をほとんど集められない**現場ではこの形しか取れない。
- ⚠️ **`score_samples` は「正常らしさ」を返す。**符号を反転しないと AUC が 0.5 を割る。上のセルで実際に確認できる。
- **`contamination` / `nu` は「データの何割が異常か」の事前の見積もり**であって、最適化して決めるパラメータではない。業務側の知識から置く。
- **閾値は「上位何 % を見るか」という予算で決まる。**これは 1-D の LOW/REVIEW/HIGH で REVIEW 帯を人手の予算から決めたのと同じ考え方である。

⚠️ ここでは**標準化していない**（画素値なので全次元が同じ単位）。単位の違う特徴を混ぜるときは、距離ベースの OneClassSVM には標準化が必須になる。